In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-rag-trace"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [5]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

# 1. 여러 문서 로드

In [6]:


# 경로
PDF_PATHS = [
    PROJECT_ROOT / "data/raw/pdf/saving_tips/1.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/2.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/3.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/4.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/5.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/6.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/7.pdf",
    PROJECT_ROOT / "data/raw/pdf/kca_report/0417_kca_split.pdf",
    PROJECT_ROOT / "data/raw/pdf/welfare/2026_hope_ladder_selected.pdf",
    PROJECT_ROOT / "data/raw/pdf/self_report/consumer_spending_report_v2.pdf",
    PROJECT_ROOT / "data/raw/txt/saving_guides.txt"
]

In [7]:
from pathlib import Path
from langchain_community.document_loaders import PDFPlumberLoader, TextLoader

all_docs = []

for path in PDF_PATHS:
    
    # 확장자 확인
    if str(path).endswith(".pdf"):
        loader = PDFPlumberLoader(str(path))
    elif str(path).endswith(".txt"):
        loader = TextLoader(str(path), encoding="utf-8")
    else:
        continue

    docs = loader.load()

    # 출처 메타데이터 추가 ⭐ 중요
    for d in docs:
        d.metadata["source"] = path.name

    all_docs.extend(docs)

print("총 문서 수:", len(all_docs))
print(all_docs[0].page_content[:500])

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

총 문서 수: 172
금융생활에 필요한 모든 정보,「파인」(fine.fss.or.kr)으로 검색하세요
“금융은 튼튼하게, 소비자는 행복하게”
보 도 참 고 자 료
보도 2017. 1. 31.(화) 조간 배포 2017. 1. 26.(목)
담당부서 금융혁신국 서정보 팀장(3145-8210) 이동춘 수석(3145-8216)
제 목 : 금융꿀팁 200선 -㉚ 사회초년생을 위한 금융꿀팁 7가지
□ 금융감독원은 국민들이 일상적인 금융거래과정에서 알아두면
유익한 실용금융정보(금융꿀팁) 200가지를 선정, 알기 쉽게 정리하여
◦ 매주 1~3가지씩 보도참고자료를 통해 안내하고
◦ 동시에 금융소비자정보 포털사이트 “파인”(fine.fss.or.kr)에도
게시하고 있음
□ 이에 따라 서른 번째 금융꿀팁으로, “사회초년생을 위한 금융꿀팁
7가지”를 별첨과 같이 안내해 드림
<별첨> 금융꿀팁 200선-㉚ 사회초년생을 위한 금융꿀팁 7가지
☞ 본 자료를 인용하여 보도할 경우에는 출처를 표기하여 주시기 바랍니다.(http://ww


# 2. 문서 split

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=500,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(all_docs)

print(f"청킹 후 문서 수: {len(split_docs)}")

청킹 후 문서 수: 172


# 3. 임베딩 + 벡터 DB

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from ragas.embeddings import LangchainEmbeddingsWrapper

base_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(split_docs, base_embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

ragas_embeddings = LangchainEmbeddingsWrapper(base_embeddings)

print(f"벡터 수: {vectorstore.index.ntotal}")

c:\Users\user\catcher\catcher-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


벡터 수: 172


C:\Users\user\AppData\Local\Temp\ipykernel_22332\3001650575.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(base_embeddings)


# 4. Retriever 설정

In [10]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# 5. 질문 → context → 답변 생성 

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def run_rag(q):
    # 1. 문서 검색
    docs = retriever.invoke(q)

    # 2. context 만들기
    context_texts = [doc.page_content[:200] for doc in docs]

    # 3. 답변 생성
    answer = llm.invoke(
        f"""
질문: {q}

질문에 대한 핵심 답만 작성하세요.
다른 주제는 절대 포함하지 마세요.
문서에 있는 내용만 사용하세요.
답변은 반드시 1문장으로 작성하세요.

문서:
{context_texts}
"""
    ).content

    # 4. 출처
    sources = [doc.metadata.get("source", "출처 없음") for doc in docs]

    return answer, context_texts, sources

In [12]:
q = "사회초년생이 절약을 위해 가장 먼저 해야 할 행동은?"

answer, contexts, sources = run_rag(q)

print("답변:")
print(answer)

print("\n출처:")
for s in sources:
    print(s)

답변:
사회초년생이 절약을 위해 가장 먼저 해야 할 행동은 계획적인 장보기를 통해 식재료를 저렴하게 구입하는 것입니다.

출처:
7.pdf
1.pdf


In [15]:
def target(inputs: dict):
    q = inputs["question"]

    answer, contexts, sources = run_rag(q)

    return {
        "answer": answer,
        "contexts": contexts
    }

In [17]:
dataset_name = "catcher-rag-dataset"

In [19]:
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def correctness_evaluator(run, example):
    answer = run.outputs["answer"]
    ground_truth = example.outputs["ground_truth"]

    prompt = f"""
다음 답변이 정답과 얼마나 일치하는지 0~1 점수로 평가해줘.

정답:
{ground_truth}

답변:
{answer}

숫자 하나만 출력해.
"""

    score = judge_llm.invoke(prompt).content.strip()

    return {
        "key": "correctness",
        "score": float(score)
    }

In [21]:
from langsmith import Client

client = Client()

dataset_name = "catcher-rag-dataset"

questions = [
    "식비를 절약하는 방법은?",
    "불필요한 소비를 줄이는 방법은?",
    "저축을 늘리는 방법은?",
    "소비 습관을 개선하는 방법은?",
    "식비를 절약하는 방법은?"
]

ground_truths = [
    "계획적인 장보기와 식재료 관리로 식비를 절약할 수 있다.",
    "필요한 것과 원하는 것을 구분하여 소비를 줄여야 한다.",
    "꾸준한 저축과 금융상품 활용이 중요하다.",
    "소비 패턴을 점검하고 불필요한 지출을 줄여야 한다.",
    "계획적인 장보기와 식재료 보존을 통해 식비를 절약할 수 있다."
]

# Dataset 생성
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Catcher LLM RAG 평가용 데이터셋"
)

# Example 등록
for q, gt in zip(questions, ground_truths):
    client.create_example(
        inputs={"question": q},
        outputs={"ground_truth": gt},
        dataset_id=dataset.id
    )

In [22]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness_evaluator]
)

View the evaluation results for experiment: 'cooked-name-72' at:
https://smith.langchain.com/o/57276aa9-ff1c-4f75-a9e3-99e44b612c81/datasets/e45e838a-1ea4-49ce-8a6e-9cbf5e3b668c/compare?selectedSessions=a5894723-b605-48b8-b2ab-ac29d333c67a




5it [00:17,  3.51s/it]


,inputs.question,outputs.answer,outputs.contexts,error,reference.ground_truth,feedback.correctness,execution_time,example_id,id
0,식비를 절약하는 방법은?,계획적인 장보기를 통해 식재료를 저렴하게 구입해야 합니다.,[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n# 절약생활의 시작은 식...,None,계획적인 장보기와 식재료 보존을 통해 식비를 절약할 수 있다.,0.8,5.065674,96084d30-f99f-42aa-8c29-19dc5e5cedf7,019dc8f4-f663-72b3-9c19-c73037d4763f
1,소비 습관을 개선하는 방법은?,소비 습관을 개선하기 위해 미니멀라이프를 실천하고 계획적인 장보기를 통해 식비를 절...,[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n미니멀라이프란 물건을 필...,None,소비 패턴을 점검하고 불필요한 지출을 줄여야 한다.,0.7,1.994354,d9f9ece0-fd1a-417a-b708-c3d7d40a380b,019dc8f5-0d4c-7830-aa6e-edb6231387d2
2,저축을 늘리는 방법은?,꾸준히 저축하여 종잣돈을 모으는 것이 필요하다.,[제목 사회초년생을 위한 금융꿀팁 7가지\n⑤ 종잣돈 모으기\n사회생활을 시작하여 ...,None,꾸준한 저축과 금융상품 활용이 중요하다.,0.9,2.160445,04ee8997-f75d-499f-9245-e42429ab3166,019dc8f5-1779-7da1-a8b1-c99de4c1bb4c
3,불필요한 소비를 줄이는 방법은?,불필요한 소비를 줄이기 위해 미니멀라이프의 마음가짐으로 필요한 만큼만 물건을 남기고...,[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n미니멀라이프란 물건을 필...,None,필요한 것과 원하는 것을 구분하여 소비를 줄여야 한다.,0.8,2.361433,530cfdd5-0cbe-4009-9b01-8ea4adce7611,019dc8f5-2436-74b0-af14-7e019c481b91
4,식비를 절약하는 방법은?,계획적인 장보기를 통해 식재료를 저렴하게 구입해야 합니다.,[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n# 절약생활의 시작은 식...,None,계획적인 장보기와 식재료 관리로 식비를 절약할 수 있다.,0.8,1.794490,7c5c85d3-6d39-4b63-83fb-92839c8715f0,019dc8f5-2f98-71c3-b83a-d9eccf90509a
